# experts4bit-qlora v0.2.0 — cross-architecture validation (Tesla T4, stock bitsandbytes)

Runs the full test suite of [pjordanandrsn/experts4bit-qlora](https://github.com/pjordanandrsn/experts4bit-qlora) on Kaggle's free Tesla T4 (Turing, cc 7.5) against a **stock `pip install bitsandbytes`** — a different GPU architecture and kernel set than the RTX A2000 (Ampere, cc 8.6) the repo's [PROVENANCE.md](https://github.com/pjordanandrsn/experts4bit-qlora/blob/main/PROVENANCE.md) numbers were measured on. Fork it and press *Run All* to reproduce.

1. environment + exact repo commit
2. full `pytest` suite (4-bit + the v0.2.0 `ExpertsNbit` 8/16-bit schemes, offload, decode paths, loader architectures)
3. the README quickstart
4. *(optional, non-blocking)* OLMoE-1B-7B stream-loaded in 4-bit — the 'it fits' claim on a free GPU


In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap,memory.total',
                      '--format=csv'], capture_output=True, text=True).stdout)
print(sys.version)


In [ ]:
REPO = 'https://github.com/pjordanandrsn/experts4bit-qlora'
BRANCH = 'release/0.2.0'
PIN_SHA = 'c5c86f3425f816a14264a8d0238314195b2282ae'
import subprocess
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, '/kaggle/working/e4b'], check=True)
SHA = subprocess.run(['git', '-C', '/kaggle/working/e4b', 'rev-parse', 'HEAD'],
                     capture_output=True, text=True, check=True).stdout.strip()
print('validating repo commit:', SHA)


In [ ]:
%pip install -q -e '/kaggle/working/e4b[test]'
import torch, bitsandbytes, transformers
print('torch', torch.__version__, '| bitsandbytes', bitsandbytes.__version__,
      '| transformers', transformers.__version__)
assert torch.cuda.is_available(), 'expected a CUDA GPU (enable the T4 accelerator)'
import sys, importlib
sys.path.insert(0, '/kaggle/working/e4b')  # editable install isn't on the live kernel's path until restart
importlib.invalidate_caches()
print('device:', torch.cuda.get_device_name(0), '| cc', torch.cuda.get_device_capability(0))


In [ ]:
# Full suite on the T4 against stock bitsandbytes. Fails the notebook if anything is red.
r = subprocess.run(['python', '-m', 'pytest', 'tests/', '-q'],
                   cwd='/kaggle/working/e4b', capture_output=True, text=True)
print(r.stdout[-4000:])
print(r.stderr[-2000:])
assert r.returncode == 0, f'test suite failed (rc={r.returncode})'


In [ ]:
# README quickstart: 4-bit primitive + per-expert LoRA, plus the v0.2.0 nbit schemes.
import torch
from experts4bit_qlora import Experts4bit, ExpertsNbit, ExpertsLoRA

gate_up = torch.randn(8, 2 * 256, 128, device='cuda') * 0.1
down    = torch.randn(8, 128, 256, device='cuda') * 0.1
base    = Experts4bit.from_float(gate_up, down, quant_type='nf4', compute_dtype=torch.float32)
model   = ExpertsLoRA(base, r=8, alpha=16).to('cuda')
hs  = torch.randn(4, 128, device='cuda')
idx = torch.randint(0, 8, (4, 2), device='cuda')
wts = torch.rand(4, 2, device='cuda')
with torch.no_grad():
    out = model(hs, idx, wts)
print('4-bit forward OK:', tuple(out.shape), out.dtype)

err = {}
for q in ('nf4', 'int8', 'bf16'):
    b = ExpertsNbit.from_float(gate_up, down, quant_type=q, compute_dtype=torch.float32)
    deq = b._dequantize_expert(b.gate_up_proj, b.gate_up_absmax, b._gate_up_shape, 0, torch.float32)
    err[q] = (deq - gate_up[0]).abs().mean().item()
print('fidelity ordering (mean abs err):', err)
assert err['bf16'] < err['int8'] < err['nf4']


## Optional: OLMoE-1B-7B fits in 4-bit on this free T4
bf16 OLMoE is ~13.9 GB — over this card's 15 GB after CUDA context, and it OOMs a 12 GB card outright. The streaming loader quantizes the fused experts to NF4 on the way in (~4.7 GB loaded). ~14 GB download; **non-blocking** — the validation above stands regardless of this cell.

In [ ]:
try:
    import torch
    from experts4bit_qlora.loader import load_moe_4bit_streaming
    model, cfg = load_moe_4bit_streaming('allenai/OLMoE-1B-7B-0924', 'cuda',
                                         torch.bfloat16, r=8, alpha=16)
    print(f'loaded OLMoE-1B-7B in 4-bit: {torch.cuda.memory_allocated()/1e9:.2f} GB on GPU')
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained('allenai/OLMoE-1B-7B-0924')
    ids = tok('The memory wall for mixture-of-experts models', return_tensors='pt').input_ids.to('cuda')
    model.eval()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=16, do_sample=False)
    print(tok.decode(out[0], skip_special_tokens=True))
except Exception as e:
    print(f'OPTIONAL CELL FAILED (non-blocking): {type(e).__name__}: {e}')


In [ ]:
print('SUMMARY-FOR-PROVENANCE')
print('repo commit:', SHA)
import torch, bitsandbytes, transformers
print('gpu:', torch.cuda.get_device_name(0), 'cc', torch.cuda.get_device_capability(0))
print('torch', torch.__version__, '| bitsandbytes', bitsandbytes.__version__,
      '| transformers', transformers.__version__)
print('suite: see pytest cell above')
